# Casablanca Traffic — profilage de la source

Ce notebook inspecte les **13 feuilles**, sans enregistrer le classeur et sans
charger la base. Les valeurs originales restent distinctes des corrections candidates.
Les mesures décrivent une semaine type, sans date réelle de collecte.
Sélectionner le kernel **CasaTraffic (Python 3.11)** et exécuter toutes les cellules.


In [1]:
import hashlib
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

assert sys.version_info[:2] == (3, 11)
assert Path(sys.prefix).name == 'CasaTraffic', sys.executable
ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.extract.workbook_profile import SOURCE_SHA256, profile_workbook

SOURCE = ROOT / 'data/source/Dataset_for_traffic_analysis_in_Casablanca__Morocco.xlsx'
initial_hash = hashlib.sha256(SOURCE.read_bytes()).hexdigest()
assert initial_hash == SOURCE_SHA256
print(sys.executable)
report, tables, measures = profile_workbook(SOURCE)
print(f"{report['sheet_count']} feuilles, {len(measures):,} observations candidates")

C:\Users\SafeyeddineK\Desktop\ProjetV2\casablanca-traffic-platform\CasaTraffic\Scripts\python.exe


13 feuilles, 73,920 observations candidates


## Structure et en-têtes

Le sommaire sert à la navigation et ne contient pas de mesures.
La ligne d'en-tête est détectée par son libellé Commune. Les heures 0–23 sont contrôlées
pour les deux blocs. La propagation des communes/ZIP se limite aux plages fusionnées.


In [2]:
overview_columns = ['sheet', 'role', 'physical_rows', 'physical_columns',
                    'header_row', 'data_rows', 'merged_ranges', 'formula_count',
                    'raw_missing_commune', 'missing_commune_after_merged_fill']
display(pd.DataFrame(report['sheets']).reindex(columns=overview_columns))

,sheet,role,physical_rows,physical_columns,header_row,data_rows,merged_ranges,formula_count,raw_missing_commune,missing_commune_after_merged_fill
0,Summary,navigation,17,6,NaN,0,2,0,NaN,NaN
1,Table 0. Coordinates,dimension_source,122,8,12.0,110,46,0,88.0,0.0
2,Table 1. Population size in eac,dimension_source,32,8,10.0,22,2,0,0.0,0.0
3,Table 2. Number of Tram and Bus,dimension_source,32,8,10.0,22,2,0,0.0,0.0
4,Table 3. Type of roads,dimension_source,32,15,10.0,22,2,0,0.0,0.0
5,Table 4. Land use variables,dimension_source,32,10,10.0,22,2,0,0.0,0.0
6,Table 5. Monday,traffic,450,60,10.0,440,51,880,418.0,0.0
7,Table 6. Tuesday,traffic,452,60,12.0,440,51,880,418.0,0.0
8,Table 7. Wednesday,traffic,452,60,12.0,440,44,880,285.0,0.0
9,Table. 8 Thursday,traffic,452,60,12.0,440,51,880,418.0,0.0


## Profil de chaque feuille

Pour chaque table : manques par colonne, statistiques numériques et outliers IQR.
Un outlier statistique reste une observation. Le compteur `non_numeric` est normal
pour des champs texte comme Coordinates ou une formule de concaténation.
Les ZIP et les indices sont des identifiants, pas des variables continues à nettoyer.


In [3]:
for sheet in report['sheets']:
    display(Markdown(f"### {sheet['sheet']}"))
    number = sheet['table_number']
    if number is None:
        display(Markdown('Sommaire : 0 ligne de mesure.'))
        continue
    display(pd.DataFrame.from_dict(sheet['missing_by_column'], orient='index',
                                   columns=['missing_after_merged_fill']))
    numeric = pd.DataFrame.from_dict(sheet['numeric_by_column'], orient='index')
    display(numeric.reindex(columns=['rows', 'missing', 'non_numeric', 'min',
                                     'median', 'p95', 'max', 'iqr_outliers']))
    display(tables[number].iloc[:3, :min(8, len(tables[number].columns))])

### Summary

Sommaire : 0 ligne de mesure.

### Table 0. Coordinates

,missing_after_merged_fill
Commune,0
ZIP code,0
Index,0
Latitude,0
Longitude,0
Coordinates,0
_excel_row,0


,rows,missing,non_numeric,min,median,p95,max,iqr_outliers
ZIP code,110,0,0,20000.000000,20425.000000,28822.000000,29640.000000,10.0
Index,110,0,0,0.000000,54.500000,103.550000,109.000000,0.0
Latitude,110,0,0,33.481472,33.568135,33.615733,33.647921,4.0
Longitude,110,0,0,-7.706392,-7.594752,-7.479561,-7.395306,2.0
Coordinates,110,0,110,NaN,NaN,NaN,NaN,NaN


...

,commune,zip,origin_index,origin_coordinates,origin_lat,origin_lon,dest_index,dest_coordinates
0,Echchallalate,28822,0,"=F13&"",""&G13",33.56968640878232,-7.395306348809435,1,"=J13&"",""&K13"
1,Echchallalate,28822,0,"=F14&"",""&G14",33.56968640878232,-7.395306348809435,2,"=J14&"",""&K14"
2,Echchallalate,28822,0,"=F15&"",""&G15",33.56968640878232,-7.395306348809435,3,"=J15&"",""&K15"


### Table. 11 Sunday

,missing_after_merged_fill
commune,0
zip,0
origin_index,0
origin_coordinates,0
origin_lat,0
origin_lon,0
dest_index,0
dest_coordinates,0
dest_lat,0
dest_lon,0


,rows,missing,non_numeric,min,median,p95,max,iqr_outliers
zip,440,0,0,20000.000000,20425.000000,28822.000000,29640.000000,40.0
origin_index,440,0,0,0.000000,54.500000,104.000000,114.000000,0.0
origin_coordinates,440,0,440,NaN,NaN,NaN,NaN,NaN
origin_lat,440,0,0,33.481472,33.568135,33.616034,33.647921,16.0
origin_lon,440,0,0,-7.706392,-7.594752,-7.475567,-7.395306,8.0
dest_index,440,0,0,0.000000,54.500000,104.000000,114.000000,0.0
dest_coordinates,440,0,440,NaN,NaN,NaN,NaN,NaN
dest_lat,440,0,0,33.481472,33.568135,33.616034,33.647921,16.0
dest_lon,440,0,0,-7.706392,-7.594752,-7.475567,-7.395306,8.0
distance_raw,440,0,0,0.617000,2652.500000,6886.350000,11574.000000,20.0


,commune,zip,origin_index,origin_coordinates,origin_lat,origin_lon,dest_index,dest_coordinates
0,Echchallalate,28822,0,"=F13&"",""&G13",33.56968640878232,-7.395306348809435,1,"=J13&"",""&K13"
1,Echchallalate,28822,0,"=F14&"",""&G14",33.56968640878232,-7.395306348809435,2,"=J14&"",""&K14"
2,Echchallalate,28822,0,"=F15&"",""&G15",33.56968640878232,-7.395306348809435,3,"=J15&"",""&K15"


## Unités de distance et de temps

Règle du plan : une distance >100 est interprétée comme des mètres et divisée par 1000.
Hypothèse supplémentaire étayée par les distributions : un temps entier >=1000
a perdu son séparateur décimal et est divisé par 1000. Aucun temps n'est dérivé du TTI.
Une correction garde la valeur brute, la valeur candidate et un indicateur explicite.


In [4]:
routes = measures.drop_duplicates(['day_of_week', '_excel_row'])
display(pd.DataFrame({
    'distance_raw': routes.distance_raw.describe(),
    'distance_km_candidate': routes.distance_km_candidate.describe(),
}))
display(pd.DataFrame({
    'time_raw': measures.travel_time_raw.describe(),
    'time_min_candidate': measures.travel_time_min_candidate.describe(),
}))
daily = measures.groupby(['day_of_week', 'day']).agg(
    observations=('hour', 'size'),
    times_scaled=('time_scaled_candidate', 'sum'),
    distance_measurements_scaled=('distance_scaled_candidate', 'sum'),
    provided_tti_below_one=('source_tti_missing_or_below_one', 'sum'),
    provided_tti_above_five=('source_tti_above_five', 'sum'),
)
display(daily)
assert len(measures) == 7 * 440 * 24
assert (daily.observations == 440 * 24).all()
display(measures.loc[measures.time_scaled_candidate,
    ['sheet', '_excel_row', 'hour', 'travel_time_raw', 'travel_time_min_candidate']].head(10))

,distance_raw,distance_km_candidate
count,3080.00000,3080.000000
mean,3020.57334,3.321858
std,2136.01659,2.001262
min,0.61700,0.617000
25%,1655.00000,1.884000
50%,2653.00000,2.892000
75%,3955.00000,4.111000
max,15169.00000,15.169000


,time_raw,time_min_candidate
count,73920.000000,73920.000000
mean,4258.622753,7.476585
std,4522.068272,3.489194
min,1.050000,1.017000
25%,7.750000,4.867000
50%,3633.000000,7.017000
75%,7533.000000,9.621000
max,26233.000000,27.383000


,,observations,times_scaled,distance_measurements_scaled,provided_tti_below_one,provided_tti_above_five
day_of_week,day,,,,,
1,monday,10560,0,9552,5,25
2,tuesday,10560,6921,9480,6,36
3,wednesday,10560,7027,9552,1,30
4,thursday,10560,7061,9336,7,38
5,friday,10560,7081,9504,6,10
6,saturday,10560,7008,9432,5,30
7,sunday,10560,7076,9480,4,2


,sheet,_excel_row,hour,travel_time_raw,travel_time_min_candidate
10560,Table 6. Tuesday,13,0,10283.0,10.283
10561,Table 6. Tuesday,14,0,11983.0,11.983
10563,Table 6. Tuesday,16,0,15017.0,15.017
10564,Table 6. Tuesday,17,0,10933.0,10.933
10565,Table 6. Tuesday,18,0,11433.0,11.433
10567,Table 6. Tuesday,20,0,15317.0,15.317
10568,Table 6. Tuesday,21,0,12183.0,12.183
10569,Table 6. Tuesday,22,0,10167.0,10.167
10570,Table 6. Tuesday,23,0,4167.0,4.167
10572,Table 6. Tuesday,25,0,10133.0,10.133


## Clés des points et distances variables

Les indices source des trajets ne peuvent pas être joints directement à Table 0.
Un crosswalk est proposé à partir des coordonnées arrondies à 8 décimales, contrôlées
pour l'unicité. Tous les originaux restent disponibles. Les distances observées par jour
seront conservées même si dim_trajectory utilise la distance du lundi comme référence.


In [5]:
identifiers = report['identifiers']
display(pd.DataFrame([
    {'raw_index': int(raw), 'coordinate_point_index': values[0]}
    for raw, values in identifiers['point_index_crosswalk'].items()
    if len(values) == 1 and int(raw) != values[0]
]))
assert not identifiers['ambiguous_source_indices']
assert identifiers['unmatched_coordinates_measurements'] == 0
variants = routes.groupby(['origin_point_candidate', 'dest_point_candidate'])[
    'distance_km_candidate'].nunique()
display(variants.value_counts().sort_index().rename_axis('distance_variants').to_frame('trajectories'))
display(pd.Series(report['dimensions']).drop('points_per_commune'))

,raw_index,coordinate_point_index
0,110,105
1,111,106
2,112,107
3,113,108
4,114,109


,trajectories
distance_variants,
1,186
2,115
3,81
4,38
5,17
6,3


commune_count                                                                                  22
commune_pairs_mismatch_by_table                              {'1': [], '2': [], '3': [], '4': []}
population_fractional_rows                                                                     14
household_fractional_rows                                                                      14
density_unit                                                        Non déclarée dans le classeur
density_differs_from_population_per_km2_rows                                                   22
density_population_area_absolute_difference     {'rows': 22, 'missing': 0, 'non_numeric': 0, '...
latitude                                        {'rows': 110, 'missing': 0, 'non_numeric': 0, ...
longitude                                       {'rows': 110, 'missing': 0, 'non_numeric': 0, ...
dtype: object

## Recalcul indépendant du TTI et comparaison

Référence choisie : minimum des temps valides corrigés sur les **168 heures du même
trajet dirigé**. TTI candidat = temps / référence. Cette référence empirique ne prouve
pas un véritable état de circulation fluide et sera figée pour le rejeu horaire.
Le TTI fourni est conservé. Un écart absolu >0,1 déclenche un signal, sans suppression.
Le diagnostic temps/distance à 60 km/h est une comparaison, pas une formule affirmée
par la source. Un TTI élevé seul ne suffit pas à déclarer une mesure invalide.


In [6]:
display(measures[['tti_provided', 'tti_recalculated_candidate',
                  'tti_delta_candidate', 'speed_kmh_candidate']].describe())
hourly = measures.groupby('hour').agg(
    max_provided=('tti_provided', 'max'),
    max_recalculated=('tti_recalculated_candidate', 'max'),
    mean_absolute_gap=('tti_delta_candidate', lambda s: s.abs().mean()),
)
display(hourly)
sequence_present = np.allclose(hourly.loc[5:23, 'max_provided'], np.arange(5, 24))
print('Suite 5, 6, ..., 23 constatée :', sequence_present)
print('Écart absolu moyen :', measures.tti_delta_candidate.abs().mean())
print('Écarts >0,1 :', int(measures.tti_absolute_gap_above_0_1.sum()))
assert measures.tti_recalculated_candidate.ge(1).all()
route_keys = ['origin_point_candidate', 'dest_point_candidate']
assert measures[route_keys].drop_duplicates().shape[0] == 440

,tti_provided,tti_recalculated_candidate,tti_delta_candidate,speed_kmh_candidate
count,73920.000000,73920.000000,73920.000000,73920.000000
mean,2.443970,1.323890,-1.120080,26.867254
std,0.693019,0.265477,0.589367,8.721337
min,0.967785,1.000000,-4.993673,7.803774
25%,1.997007,1.142620,-1.490312,21.042283
50%,2.413080,1.261519,-1.137821,24.923393
75%,2.852499,1.432909,-0.744337,30.235821
max,7.688588,4.548759,1.208358,93.714684


,max_provided,max_recalculated,mean_absolute_gap
hour,,,
0,4.849119,2.871643,1.066036
1,5.128849,2.877572,1.073075
2,5.383023,2.877572,1.100369
3,5.053548,2.877572,1.116048
4,5.295725,2.877572,1.130004
5,5.110897,2.877572,1.123661
6,5.110897,2.877572,1.123159
7,5.102041,2.877572,1.106857
8,7.095710,3.721870,1.133420


Suite 5, 6, ..., 23 constatée : False
Écart absolu moyen : 1.125585700278673
Écarts >0,1 : 70810


## Conservation et limites

Aucune ligne n'a été supprimée et aucune table métier n'a encore été chargée.
Les règles seront implémentées et validées dans les phases 3–5 ; une mesure
irréparable devra rejoindre quarantine avec son motif. La densité source n'a pas
d'unité déclarée, les populations/ménages fractionnaires sont conservés.
Les outliers IQR ne sont pas des règles de rejet. Aucun lien causal n'est inféré.


In [7]:
final_hash = hashlib.sha256(SOURCE.read_bytes()).hexdigest()
assert final_hash == initial_hash == SOURCE_SHA256
assert len(report['sheets']) == 13
assert report['dimensions']['commune_count'] == 22
assert len(tables[0]) == 110
assert all(len(tables[n]) == 440 for n in range(5, 12))
print('Source intacte et cardinalités vérifiées. Notebook exécuté avec CasaTraffic.')

Source intacte et cardinalités vérifiées. Notebook exécuté avec CasaTraffic.
